# 6. Multi-Agent Systems — The Supervisor Pattern

## Why multiple agents?

One agent with 15 tools and a 2-page prompt gets confused. Splitting work across
**specialized agents** keeps each prompt small and focused — the same reason companies
have departments instead of one employee doing everything.

## The supervisor pattern

A **supervisor** node decides who works next; **workers** do the work and report back:

```
            ┌──────────────┐
    ┌──────▶│  supervisor  │◀───────┐
    │       └──┬───────┬───┘        │
    │          ▼       ▼            │
researcher   writer   editor ───────┘
    (each worker returns to the supervisor)
```

New tool for this: **`Command(goto=...)`** — a node can return a `Command` that both
updates state AND names the next node, replacing separate router functions.

## Real-life example: content studio

A marketing team producing a blog post:

- **researcher** — gathers key facts and angles on the topic
- **writer** — turns research (and any editor feedback) into a draft
- **editor** — reviews the draft: APPROVE or REVISE with feedback
- **supervisor** — coordinates: research first, then write, then edit; on REVISE,
  send it back to the writer (max 2 revisions so we never loop forever)

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
# .env in this repo stores the key as GOOGLE_API_KEY_1 — normalize it
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY") or os.getenv("GOOGLE_API_KEY_1")

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")
llm.invoke("Say 'ready' if you can hear me.").content

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command


class StudioState(TypedDict):
    topic: str
    research_notes: str
    draft: str
    editor_feedback: str
    verdict: str          # "", "APPROVE", or "REVISE"
    revision_count: int

In [ ]:
MAX_REVISIONS = 2


def supervisor(state: StudioState) -> Command[Literal["researcher", "writer", "editor", "__end__"]]:
    # Deterministic coordination logic. (You could also let an LLM decide —
    # rule-based supervisors are cheaper and predictable; LLM supervisors are flexible.)
    if not state.get("research_notes"):
        nxt, why = "researcher", "no research yet"
    elif not state.get("draft"):
        nxt, why = "writer", "research done, no draft yet"
    elif not state.get("verdict"):
        nxt, why = "editor", "draft ready, needs review"
    elif state["verdict"] == "REVISE" and state["revision_count"] < MAX_REVISIONS:
        nxt, why = "writer", f"editor wants changes (revision {state['revision_count'] + 1})"
    else:
        nxt, why = END, f"verdict={state['verdict']}, done"
    print(f"SUPERVISOR -> {nxt}  ({why})")
    return Command(goto=nxt)

In [ ]:
def researcher(state: StudioState) -> Command[Literal["supervisor"]]:
    prompt = f'''You are a research assistant. List 5 key facts, statistics, or angles
about: {state["topic"]}. Bullet points, one line each.'''
    notes = llm.invoke(prompt).content
    return Command(goto="supervisor", update={"research_notes": notes})


def writer(state: StudioState) -> Command[Literal["supervisor"]]:
    feedback_part = ""
    revisions = state.get("revision_count", 0)
    if state.get("editor_feedback"):
        feedback_part = f'''
The editor reviewed your previous draft and said:
{state["editor_feedback"]}
Previous draft:
{state["draft"]}
Rewrite it addressing every point.'''
        revisions += 1
    prompt = f'''You are a blog writer. Write a punchy ~200-word blog post about
"{state["topic"]}" using this research:
{state["research_notes"]}{feedback_part}'''
    draft = llm.invoke(prompt).content
    # Reset verdict so the editor reviews the NEW draft
    return Command(goto="supervisor",
                   update={"draft": draft, "verdict": "", "revision_count": revisions})


def editor(state: StudioState) -> Command[Literal["supervisor"]]:
    prompt = f'''You are a strict editor. Review this draft about "{state["topic"]}".
First line: exactly APPROVE or REVISE.
If REVISE, follow with 2-3 specific improvement points.

{state["draft"]}'''
    review = llm.invoke(prompt).content
    verdict = "APPROVE" if review.strip().upper().startswith("APPROVE") else "REVISE"
    return Command(goto="supervisor",
                   update={"verdict": verdict, "editor_feedback": review})

In [ ]:
builder = StateGraph(StudioState)
builder.add_node("supervisor", supervisor)
builder.add_node("researcher", researcher)
builder.add_node("writer", writer)
builder.add_node("editor", editor)

builder.add_edge(START, "supervisor")
# No other add_edge calls needed: every node returns Command(goto=...),
# which defines the edges dynamically.

app = builder.compile()

In [ ]:
# Visualize the graph (needs internet for mermaid rendering — safe to skip)
from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Could not render image, here is the mermaid source instead:\n")
    print(app.get_graph().draw_mermaid())

In [ ]:
result = app.invoke({
    "topic": "Why small businesses should automate customer support with AI",
    "research_notes": "", "draft": "", "editor_feedback": "",
    "verdict": "", "revision_count": 0,
})

print("\n" + "=" * 70)
print(f"Verdict: {result['verdict']}  |  Revisions: {result['revision_count']}")
print("=" * 70 + "\n")
print(result["draft"])

## Streaming: watch the team work

`.invoke()` waits for the end. `.stream()` yields each node's output as it happens —
this is how you build live UIs ("Researcher is working…").

In [ ]:
for step in app.stream({
    "topic": "The rise of weekend side projects among developers",
    "research_notes": "", "draft": "", "editor_feedback": "",
    "verdict": "", "revision_count": 0,
}):
    for node_name, update in step.items():
        keys = list(update.keys()) if isinstance(update, dict) else update
        print(f"[{node_name}] updated: {keys}")

## Key takeaways

- **Supervisor** owns coordination; **workers** own one skill each. Small prompts win.
- `Command(goto=..., update=...)` = routing + state update in one return value.
- Always add a **loop guard** (`MAX_REVISIONS`) — two LLMs can disagree forever.
- The supervisor here is rule-based; swap in an LLM call when routing needs judgment.
- Each worker could itself be a full tool-using agent from notebook 4 — or a whole
  **subgraph**. That's how large systems compose.

## Where to go from here

- **Subgraphs** — nest a compiled graph as a node of another graph
- **`langgraph-supervisor` / `langgraph-swarm`** — prebuilt multi-agent libraries
- **Streaming tokens** — `stream_mode="messages"` for ChatGPT-style typing
- **LangGraph Platform / Studio** — deploy graphs as APIs, visual debugger
- Combine everything: a multi-agent system (nb 6) with tools (nb 4), memory (nb 3),
  and human approval gates (nb 5) is a production AI application.